<div class="alert alert-block alert-success">
    <h1 align="center">Customer Churn Prediction</h1>
    <h3 align="center">Machine Learning Project</h3>
</div>

<img src="https://slitayem.github.io/img/blog/2020-08-04/churn.png" width=60%>

## 1. Importing the Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
import xgboost as xgb
import lightgbm as lgb
import warnings
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv('Churn.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nTarget Distribution:\n{df['Exited'].value_counts(normalize=True)}")

In [ ]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
print(f"Shape after dropping ID columns: {df.shape}")

## 3. EDA - Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')

plt.subplot(1, 2, 2)
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].plot(kind='barh')
plt.title('Missing Values Percentage')
plt.xlabel('Percentage (%)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(x='Exited', data=df, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Churn Distribution')
axes[0, 0].set_xlabel('Exited')
axes[0, 0].set_ylabel('Count')

sns.countplot(x='Geography', hue='Exited', data=df, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Churn by Geography')

sns.countplot(x='Gender', hue='Exited', data=df, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Churn by Gender')

sns.countplot(x='NumOfProducts', hue='Exited', data=df, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Churn by Number of Products')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x='CreditScore', hue='Exited', kde=True, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Credit Score Distribution by Churn')

sns.histplot(data=df, x='Age', hue='Exited', kde=True, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Age Distribution by Churn')

sns.histplot(data=df, x='Balance', hue='Exited', kde=True, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Balance Distribution by Churn')

sns.histplot(data=df, x='EstimatedSalary', hue='Exited', kde=True, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Estimated Salary Distribution by Churn')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x='Exited', y='CreditScore', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Credit Score by Churn')

sns.boxplot(x='Exited', y='Age', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Age by Churn')

sns.boxplot(x='Exited', y='Balance', data=df, ax=axes[2], palette='Set2')
axes[2].set_title('Balance by Churn')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
df_encoded = df.copy()

le_gender = LabelEncoder()
df_encoded['Gender'] = le_gender.fit_transform(df_encoded['Gender'])

df_encoded = pd.get_dummies(df_encoded, columns=['Geography'], drop_first=True)
print(f"Shape after encoding: {df_encoded.shape}")
df_encoded.head()

In [ ]:
X = df_encoded.drop('Exited', axis=1)
y = df_encoded['Exited']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:\n{y.value_counts()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

## 5. Storytelling - Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

churn_by_gender = df.groupby(['Gender', 'Exited']).size().unstack()
churn_by_gender.plot(kind='bar', stacked=True, ax=axes[0, 0], color=['#2ecc71', '#e74c3c'])
axes[0, 0].set_title('Churn Distribution by Gender')
axes[0, 0].set_xticklabels(['Female', 'Male'], rotation=0)
axes[0, 0].legend(['Stayed', 'Churned'])

churn_by_geo = df.groupby(['Geography', 'Exited']).size().unstack()
churn_by_geo.plot(kind='bar', stacked=True, ax=axes[0, 1], color=['#2ecc71', '#e74c3c'])
axes[0, 1].set_title('Churn Distribution by Geography')
axes[0, 1].legend(['Stayed', 'Churned'])

churn_by_products = df.groupby(['NumOfProducts', 'Exited']).size().unstack()
churn_by_products.plot(kind='bar', stacked=True, ax=axes[1, 0], color=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Churn Distribution by Number of Products')
axes[1, 0].legend(['Stayed', 'Churned'])

churn_by_active = df.groupby(['IsActiveMember', 'Exited']).size().unstack()
churn_by_active.plot(kind='bar', stacked=True, ax=axes[1, 1], color=['#2ecc71', '#e74c3c'])
axes[1, 1].set_title('Churn Distribution by Active Member Status')
axes[1, 1].set_xticklabels(['Inactive', 'Active'], rotation=0)
axes[1, 1].legend(['Stayed', 'Churned'])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_bins = [18, 30, 40, 50, 60, 70, 80, 100]
df['AgeGroup'] = pd.cut(df['Age'], bins=age_bins, labels=['18-30', '31-40', '41-50', '51-60', '61-70', '71-80', '80+'])
churn_by_age = df.groupby(['AgeGroup', 'Exited']).size().unstack()
churn_by_age.plot(kind='bar', stacked=True, ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution by Age Group')
axes[0].legend(['Stayed', 'Churned'])

balance_bins = [0, 50000, 100000, 150000, 200000, 250000]
df['BalanceGroup'] = pd.cut(df['Balance'], bins=balance_bins, 
                           labels=['0-50K', '50K-100K', '100K-150K', '150K-200K', '200K+'])
churn_by_balance = df.groupby(['BalanceGroup', 'Exited']).size().unstack()
churn_by_balance.plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Distribution by Balance Group')
axes[1].legend(['Stayed', 'Churned'])

plt.tight_layout()
plt.show()

In [ ]:
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X_train_scaled, y_train)
importances = pd.Series(rf_temp.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.plot(kind='barh', color='teal')
plt.title('Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Prepare Data for Machine Learning

In [ ]:
top_features = importances.head(10).index.tolist()
X_train_selected = X_train_scaled[top_features]
X_test_selected = X_test_scaled[top_features]

print(f"Selected features ({len(top_features)}): {top_features}")
print(f"Training set shape: {X_train_selected.shape}")
print(f"Test set shape: {X_test_selected.shape}")

## 7. Train Your Model

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        use_label_encoder=False, eval_metric='logloss'
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        verbose=-1
    )
}

trained_models = {}
cv_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    scores = cross_val_score(model, X_train_selected, y_train, cv=5,
                            scoring='roc_auc', n_jobs=-1)
    cv_results[name] = {'mean_auc': scores.mean(), 'std_auc': scores.std()}
    print(f"  CV AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")
    model.fit(X_train_selected, y_train)
    trained_models[name] = model

## 8. Test the Model and Show the Metrics

In [ ]:
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_selected)
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'Accuracy': accuracy, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'ROC AUC': roc_auc,
        'predictions': y_pred, 'probabilities': y_pred_proba
    }
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  ROC AUC: {roc_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(['Logistic Regression', 'Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, results[name]['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'{name}\nAccuracy: {results[name]["Accuracy"]:.4f}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['probabilities'])
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {res['ROC AUC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
metrics_df = pd.DataFrame({name: {k: v for k, v in res.items() 
                                  if k not in ['predictions', 'probabilities']}
                           for name, res in results.items()}).T
print("Model Comparison:")
print(metrics_df.to_string())

In [ ]:
metrics_df.plot(kind='bar', figsize=(14, 6))
plt.title('Model Metrics Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['ROC AUC'])
best_model = trained_models[best_model_name]
print(f"Best Model: {best_model_name} (ROC AUC: {results[best_model_name]['ROC AUC']:.4f})")

In [ ]:
print(f"\nClassification Report for {best_model_name}:")
print(classification_report(y_test, results[best_model_name]['predictions']))

## 9. Save Your Final Model

In [ ]:
joblib.dump(best_model, 'best_churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

model_info = {
    'model_name': best_model_name,
    'features': top_features,
    'accuracy': results[best_model_name]['Accuracy'],
    'roc_auc': results[best_model_name]['ROC AUC']
}
pd.DataFrame([model_info]).to_csv('model_info.csv', index=False)

print("Model saved as 'best_churn_model.pkl'")
print("Scaler saved as 'scaler.pkl'")
print("Model info saved as 'model_info.csv'")

In [ ]:
loaded_model = joblib.load('best_churn_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
X_test_loaded = loaded_scaler.transform(X_test[top_features])
verify_pred = loaded_model.predict(X_test_loaded)
print(f"Verification Accuracy: {accuracy_score(y_test, verify_pred):.4f}")